# A100: improve your saved orchestra, then evaluate joint RL

Upload **a100_music_upgrade_v3.zip** beside this notebook. Select an
**A100 40 GB or 80 GB** CUDA runtime, Python 3.10–3.12, and persistent
results storage. No H100-name workaround cell is needed. The same GPU
BF16 check also accepts H100/H200; this release is prepared for your A100.

Starts from your **actual phrase step 280** (`progress (1).pt`). Your
listener-favorite step-128 checkpoint and original download remain archived.
The 19 roles share seven family LoRA adapters over the frozen 233M Transformer.

**Stage 1:** bounded supervised partner-response adaptation, 512 updates,
eight phrases per role/update, LR 3e-5. First 16 updates train only the
partner-condition network. Later updates train adapters, planner and conditions.
Validation every 32 updates rejects individual-role regressions and checks
preservation of phrase prediction. No checkpoint is declared better by loss alone.

**Stage 2:** 19-role joint PPO code is included, with a causal sequence
critic and checkpoints. **It is blocked in this release:** the earlier reward
experiment failed and fresh-composition human preferences are missing. The
RL cells report the reason and do not train until an independently validated,
hash-bound reward is supplied. See `A100_RL_REWARD_CONTRACT.md`.

Keep the pretrained backbone frozen. This is new partner adaptation from
step 280, not an exact continuation of your original phrase optimizer.
Audited data still uses four-bar targets with two preceding bars; more GPU
memory does not create new scores or trained long-form structure.

### v3 audit outcome and actual limits
The frozen audio reward was rechecked on six original 19-role generated clips.
It passed 12/12 loop/shuffle contrasts and gain invariance, but ranked silence
above 5/6 originals and contradicted the one explicit listener preference.
It is rejected as a taste reward. See `reward_audit/acceptance.json` and rate
the original clips at `reward_audit/listen.html`. These six ratings can help
collect preferences; they cannot alone establish generalization.

PPO now performs policy/critic updates on CUDA when available, records entropy,
reference KL, clipping and critic explained variance, uses eight independent
performances per update, and selects checkpoints on twelve fixed validation
seeds with per-role participation guards and paired uncertainty. Training reward
does not select checkpoints. The selected candidate still needs listening.

Remaining work is explicit: no accepted reward artifact is shipped, Transformer
rollouts are still sequential, and the reward/advantages are still shared team
signals. Counterfactual credit, batched Transformer rollouts and planner PPO
adaptation are not claimed as implemented. Full A100 execution is unverified.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json, hashlib, shutil, time, signal
from IPython.display import display, Audio, Image, FileLink, clear_output

# EDIT these paths for your provider. ZIP_PATH is the archive you upload.
ZIP_PATH = Path.cwd() / "a100_music_upgrade_v3.zip"
EXTRACT_ROOT = Path.cwd() / "music_workspace"
# IMPORTANT: set this to a mounted persistent disk / Drive folder if available.
RESULTS_ROOT = Path(os.environ.get("MUSIC_RUN_ROOT", str(Path.cwd() / "music_results")))
PROJECT = EXTRACT_ROOT / "a100_music_package"
RUN = RESULTS_ROOT / "a100_partner_280_v3"
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("Select a Python 3.10–3.12 notebook runtime for the pinned environment.")
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Upload the training ZIP here or edit ZIP_PATH: {ZIP_PATH}")
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    for info in archive.infolist():
        target = (EXTRACT_ROOT / info.filename).resolve()
        if not target.is_relative_to(EXTRACT_ROOT.resolve()):
            raise ValueError("Unsafe archive path")
    # Reuse an existing extraction; do not overwrite a run or modified files.
    if not PROJECT.exists():
        archive.extractall(EXTRACT_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(PROJECT / "h100_assets.py"), "verify"], check=True)
print("Project:", PROJECT)
print("Results:", RESULTS_ROOT, "— confirm this storage survives a notebook restart.")
INIT = PROJECT / "checkpoints/a100_step280.pt"
FAVORITE = PROJECT / "checkpoints/listener_favorite_128.pt"

## Install the tested environment
The isolated venv keeps the notebook provider's Python packages unchanged.
PyTorch 2.6 CUDA 12.4 is a compatibility pin, not a claim that it is the newest
version. A recent NVIDIA driver is required; CUDA Toolkit installation is not
needed separately. This installation may take several minutes.

In [ ]:
VENV = EXTRACT_ROOT / "a100_music_venv"
PY = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not PY.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
marker = VENV / "music_dependencies_v2.ready"
if not marker.exists():
    subprocess.run([str(PY), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([str(PY), "-m", "pip", "install", "torch==2.6.0",
                    "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
    subprocess.run([str(PY), "-m", "pip", "install", "-r", str(PROJECT / "requirements-h100.txt")], check=True)
    marker.write_text("isolated environment installed")
ENV = dict(os.environ, USE_TF="0", USE_FLAX="0", HF_HUB_DISABLE_XET="1", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg")
def run_command(args):
    return subprocess.run([str(PY), *map(str, args)], cwd=PROJECT, env=ENV, check=True)
gpu_probe = ("import json, torch; ok = torch.cuda.is_available(); "
             "print(json.dumps(dict(torch=torch.__version__, cuda_available=ok, "
             "name=torch.cuda.get_device_name(0) if ok else 'CPU', "
             "vram_bytes=torch.cuda.get_device_properties(0).total_memory if ok else 0, "
             "bf16=torch.cuda.is_bf16_supported() if ok else False)))")
result = subprocess.run([str(PY), "-c", gpu_probe], cwd=PROJECT, env=ENV,
                        check=True, capture_output=True, text=True)
GPU_INFO = json.loads(result.stdout)
if not GPU_INFO["cuda_available"] or not GPU_INFO["bf16"]:
    raise RuntimeError(f"This recipe requires a CUDA GPU with BF16 support. Assigned device: {GPU_INFO}")
# The v1 trainer already supports other GPUs through this explicit override.
ACCELERATOR_ARGS = [] if "H100" in GPU_INFO["name"] and GPU_INFO["vram_bytes"] >= 70 * 1024**3 else ["--allow-non-h100"]
print("Assigned GPU:", GPU_INFO)
print("Batch calibration will use this GPU's actual memory and throughput.")
run_command(["h100_assets.py", "download"])
# Native Linux sound renderer for the later listening cells.
if os.name != "nt" and (not shutil.which("fluidsynth") or not shutil.which("ffmpeg")):
    if hasattr(os, "geteuid") and os.geteuid() == 0 and shutil.which("apt-get"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "fluidsynth", "ffmpeg", "libsndfile1"], check=True)
    else:
        raise RuntimeError("Install fluidsynth, ffmpeg and libsndfile1 using your provider's system package manager, then rerun this cell.")
print("Environment and pinned assets ready.")

## Verify the data and production sampler
The smoke test performs a real optimizer update covering every role, checks
gradient flow with the backbone frozen, reloads weights/optimizer/RNG, and
samples a draft that reaches the score boundary. It uses a separate output
directory and never changes the recommended training initialization.

In [ ]:
run_command(["-m", "unittest", "test_h100_package", "test_orchestra_learning", "test_full_orchestra", "test_orchestra_joint_rl", "test_orchestra_rl_selection", "-q"])
SMOKE = RESULTS_ROOT / "smoke"
if not (SMOKE / "smoke_result.json").exists():
    run_command(["h100_train.py", "--mode", "smoke", "--output", SMOKE,
                 "--init", INIT, "--steps", "2", "--phrases-per-role", "1", "--microbatch", "16",
                 "--validation-groups", "1", *ACCELERATOR_ARGS])
smoke = json.loads((SMOKE / "smoke_result.json").read_text())
assert smoke["passed"]
print("GPU verification:", GPU_INFO["name"], "| H100 execution:", smoke["h100_executed"])
print(smoke)

## Choose a bounded partner run
512 partner updates, eight phrases per role, a six-hour cumulative budget. The
first 16 updates warm up conditioning while adapters and planner stay frozen.
Then all three supervised components adapt together. A100 calibration changes
physical chunks only; it keeps every selected training example and the effective
phrase count. Every eight updates save weights, optimizer, RNG and recipe hashes.

The audit compares exact step-280 weights against the listener-favorite checkpoint
before any optimizer updates. Partner selection must improve balanced event loss
and keep each role's event and attribute losses within 2% of its starting value;
the same per-role preservation check runs without announced drafts. Failed
adaptation falls back to the earlier checkpoint. Start a new folder to change
learning rate or sampling; resume requires the original recipe.

In [ ]:
STEPS = 512  # partner updates; starts from the saved A100 phrase step 280
PHRASES_PER_ROLE = 8  # unchanged effective sampling: 8 phrases x 19 roles
HOURS = 6  # cumulative; includes audits, calibration and draft generation
LEARNING_RATE = 3e-5  # new adaptation stage, intentionally new optimizer
MICROBatch = 0  # calibrate on the actual A100, without changing effective batch
TOKEN_BUDGET = 131072
BASE_ARGS = ["h100_train.py", "--init", INIT, "--output", RUN,
    "--start-stage", "partner", "--partner-warmup", "16", "--preserve-phrase",
    "--steps", str(STEPS), "--phrases-per-role", str(PHRASES_PER_ROLE),
    "--hours", str(HOURS), "--learning-rate", str(LEARNING_RATE),
    "--microbatch", str(MICROBatch), "--token-budget", str(TOKEN_BUDGET),
    "--validation-interval", "32", "--save-interval", "8", "--draft-refresh", "64",
    "--seed", "9932001", *ACCELERATOR_ARGS]
if not (RUN / "verification.json").exists():
    verify_args = BASE_ARGS + ["--mode", "verify"]
    if (RUN / "progress.pt").exists(): verify_args += ["--resume"]
    run_command(verify_args)
if not (RUN / "batch_benchmark.json").exists():
    run_command(BASE_ARGS + ["--mode", "benchmark", "--resume"])
if not (RUN / "initial_audit.json").exists():
    current = json.loads((RUN / "training_log.json").read_text())
    if current.get("updates"):
        raise RuntimeError("Missing initial audit on a previously updated run; retain this run and use a fresh folder.")
    run_command(BASE_ARGS + ["--mode", "audit", "--audit-reference", FAVORITE, "--resume"])
display(json.loads((RUN / "batch_benchmark.json").read_text()))
print("Step 280 is the starting checkpoint. Step 256 weights were not present in your download.")
print("The original optimizer/RNG are archived; this is a new partner-adaptation stage.")

## Start / resume training
This starts a child process and returns immediately. The monitor in the next
cell shows its actual logs and charts. Interrupting the **monitor** does not
stop training. Use the explicit stop cell to checkpoint and stop the trainer.
Do not close the runtime until the results are on persistent storage or downloaded.

In [ ]:
import platform
PID_FILE = RUN / "trainer.pid"
if PID_FILE.exists():
    old_pid = int(PID_FILE.read_text())
    try:
        os.kill(old_pid, 0)
    except (ProcessLookupError, OSError):
        PID_FILE.unlink()
    else:
        raise RuntimeError(f"PID {old_pid} is still alive. Do not start a duplicate trainer; use the monitor or stop cell.")
if json.loads((RUN / "training_log.json").read_text()).get("complete"):
    print("This run is already complete; continue to listening evaluation.")
else:
    stream = (RUN / "console.log").open("a", buffering=1)
    trainer = subprocess.Popen([str(PY), "-u", *map(str, BASE_ARGS), "--mode", "train", "--resume"],
                               cwd=PROJECT, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
    PID_FILE.write_text(str(trainer.pid))
    print("Training PID:", trainer.pid, "Log:", RUN / "console.log")

In [ ]:
# Run this again whenever you want progress. Interrupt only this cell to stop monitoring.
while True:
    try:
        state = json.loads((RUN / "training_log.json").read_text())
    except (OSError, json.JSONDecodeError):
        time.sleep(2); continue
    clear_output(wait=True)
    print(state["status"], "|", state["stage"], state["step"], "/", state["total"],
          "|", round(state["elapsed"] / 60, 1), "minutes")
    print("Selected validation update:", state.get("selected_step", 0), "| final stages:", state.get("selections", {}))
    if state.get("updates"):
        last = state["updates"][-1]
        print("Loss:", round(last["loss"], 4), "Examples:", last["examples"],
              "Physical batch:", last["physical_microbatch"], "Step seconds:", round(last["seconds"], 2))
    run_command(["h100_plot.py", "--run", RUN])
    display(Image(filename=str(RUN / "training_progress.png")))
    log_file = RUN / "console.log"
    if log_file.exists(): print("\n".join(log_file.read_text(errors="replace").splitlines()[-8:]))
    if state.get("complete") or state.get("failure"): break
    if "trainer" in globals() and trainer.poll() is not None:
        print("Training process exited:", trainer.returncode); break
    time.sleep(15)

### Optional: stop at a complete update boundary
Run only when you intend to stop. Then rerun the start/resume cell to continue.
The trainer catches SIGTERM/SIGINT; providers may forcibly kill it if they do
not allow enough shutdown time. Periodic checkpoints remain the fallback.

In [ ]:
# Intentionally disabled so Run All does not stop your training.
STOP_TRAINING = False
if STOP_TRAINING and PID_FILE.exists():
    os.kill(int(PID_FILE.read_text()), signal.SIGTERM)
    print("Stop requested. Wait for the trainer to exit before resuming.")

## Listen and probe actual partner use
Run after training completes. The first fixed seed gets the full held-out
partner probes. The other fixed seeds are generated with no cherry-picking.
Increase the predetermined seed list to 12 for a broader audit. Same seed does
not imply identical music; it controls the random draws while models differ.

Both initial and trained conditions use identical quiet rendering and profile
settings. The profile is an inference control, not evidence of learned happiness.
If strict checkpoint selection fails, the page explicitly labels the candidate
experimental. There is no automatic preference or reward approval.

In [ ]:
state = json.loads((RUN / "training_log.json").read_text())
assert state.get("complete"), "Resume/finish training first."
EVAL_SEEDS = [9919101, 9919102, 9919103]  # Set BEFORE listening; retain every output.
EVAL_ROOT = RESULTS_ROOT / "listening"
for i, seed in enumerate(EVAL_SEEDS):
    destination = EVAL_ROOT / str(seed)
    command = ["evaluate_trained_orchestra.py", "--run", RUN,
               "--output", destination, "--seed", str(seed), "--initial-checkpoint", INIT]
    if i: command += ["--skip-probes"]
    run_command(command)
    print("Seed", seed, "before training")
    display(Audio(filename=str(destination / "initial.mp3")))
    print("Seed", seed, "after training")
    display(Audio(filename=str(destination / "trained.mp3")))
    display(FileLink(str(destination / "listen.html")))

## Joint multi-agent RL: readiness check
The legacy three-instrument PPO is not used here. `orchestra_joint_rl.py`
supports all 19 role IDs, independent residual note actors and a shared
sequence critic. It samples private frozen drafts and commits final notes
synchronously. All note attributes and rests remain policy decisions.

The shipped reward acceptance record **failed**. This check runs before
allocating a model or changing policy weights. Do not change the booleans
to force a pass. `reward.ts` and independent validation evidence are still
needed; the contract documents inputs and exact generator/code hashes.

In [ ]:
RL_RUN = RESULTS_ROOT / "a100_joint_rl_v3"
RL_SOURCE = RUN / "selected.pt"
REWARD_DIR = PROJECT / "provenance/reward_validation"
RL_ARGS = ["orchestra_joint_rl.py", "--source", RL_SOURCE, "--reward", REWARD_DIR,
           "--output", RL_RUN, "--updates", "24", "--episodes", "8", "--bars", "8", "--hours", "2"]
if RL_SOURCE.exists():
    result = subprocess.run([str(PY), *map(str, RL_ARGS), "--check"], cwd=PROJECT,
                            env=ENV, capture_output=True, text=True, check=True)
    RL_READINESS = json.loads(result.stdout.strip().splitlines()[-1])
    print(json.dumps(RL_READINESS, indent=2))
else:
    RL_READINESS = {"rl_allowed": False, "reason": "Finish partner training and checkpoint selection first."}
    print(RL_READINESS)
print("Contract:", PROJECT / "A100_RL_REWARD_CONTRACT.md")

### Optional real-GPU rollout verification
Disabled during Run All. It performs a two-bar joint rollout and probability
replay with zero optimizer updates. The zero score is an inference test,
never a musical-quality reward or an accepted way to start PPO.

In [ ]:
RUN_RL_ROLLOUT_SMOKE = False
if RUN_RL_ROLLOUT_SMOKE:
    assert RL_SOURCE.exists(), "Finish partner selection first."
    run_command(["orchestra_joint_rl.py", "--source", RL_SOURCE,
                 "--output", RESULTS_ROOT / "rl_rollout_smoke", "--rollout-smoke"])
    display(json.loads((RESULTS_ROOT / "rl_rollout_smoke/rollout_smoke.json").read_text()))

### Start / resume PPO only after validated reward integration
Default is off. Enabling this does not override reward validation. The
trainer checks acceptance and source/reward/code hashes again before any
GPU load or update. No automatic promotion by training reward: listen to
all fixed comparisons and test actual partner use before replacing a model.

In [ ]:
START_RL = False
if START_RL:
    if not json.loads((RUN / "training_log.json").read_text()).get("complete"):
        raise RuntimeError("Wait for the partner trainer to exit before starting PPO.")
    # Readiness is checked inside the child even if artifacts changed since the check cell.
    if not RL_READINESS.get("rl_allowed"):
        raise RuntimeError(RL_READINESS["reason"])
    RL_RUN.mkdir(parents=True, exist_ok=True)
    RL_PID = RL_RUN / "trainer.pid"
    if RL_PID.exists():
        try: os.kill(int(RL_PID.read_text()), 0)
        except (ProcessLookupError, OSError): RL_PID.unlink()
        else: raise RuntimeError("An RL trainer is already alive; use its monitor.")
    if (RL_RUN / "training_log.json").exists() and json.loads((RL_RUN / "training_log.json").read_text()).get("complete"):
        print("PPO run is already complete; use the comparison cell.")
    else:
        rl_log = (RL_RUN / "console.log").open("a", buffering=1)
        resume_args = ["--resume"] if (RL_RUN / "progress.pt").exists() else []
        rl_trainer = subprocess.Popen([str(PY), "-u", *map(str, RL_ARGS), *resume_args],
            cwd=PROJECT, env=ENV, stdout=rl_log, stderr=subprocess.STDOUT)
        RL_PID.write_text(str(rl_trainer.pid))
        print("PPO PID:", rl_trainer.pid, "|", RL_RUN / "console.log")
else:
    print("RL not started. Partner training and listening do not require the failed reward.")

In [ ]:
# Live PPO monitor; interrupting this cell does not stop training.
if "rl_trainer" in globals():
    while True:
        clear_output(wait=True)
        path = RL_RUN / "training_log.json"
        if path.exists():
            try: rl_state = json.loads(path.read_text())
            except json.JSONDecodeError: time.sleep(2); continue
            print(rl_state["status"], "| update", rl_state["step"], "| minutes", round(rl_state["elapsed"]/60, 1))
            if rl_state["updates"]:
                last = rl_state["updates"][-1]
                print("Reward:", last["reward"], "Critic loss:", last["losses"]["critic_loss"])
                print("Every role's policy loss:", {r:v["loss"] for r,v in last["losses"].items() if isinstance(v,dict) and "loss" in v})
            if rl_state.get("complete"): break
        if rl_trainer.poll() is not None:
            print("RL exited:", rl_trainer.returncode)
            print((RL_RUN / "console.log").read_text(errors="replace")[-4000:]); break
        time.sleep(15)

In [ ]:
STOP_RL = False  # Run All must not stop a trainer
if STOP_RL and (RL_RUN / "trainer.pid").exists():
    os.kill(int((RL_RUN / "trainer.pid").read_text()), signal.SIGTERM)
    print("Stop requested; last complete update is saved. Allow the current episode to finish.")

In [ ]:
if (RL_RUN / "training_log.json").exists() and json.loads((RL_RUN / "training_log.json").read_text()).get("complete"):
    for seed in [9919101, 9919102, 9919103]:
        destination = RESULTS_ROOT / "rl_listening" / str(seed)
        if not (destination / "evaluation.json").exists():
            run_command(["compare_orchestra_rl.py", "--source", RL_SOURCE, "--rl", RL_RUN / "progress.pt",
                         "--output", destination, "--seed", str(seed)])
        print("Seed", seed, "supervised foundation"); display(Audio(filename=str(destination / "supervised.mp3")))
        print("Seed", seed, "PPO candidate"); display(Audio(filename=str(destination / "rl_candidate.mp3")))
        display(FileLink(str(destination / "listen.html")))
else:
    print("No completed PPO run to compare. The accepted supervised checkpoint remains available.")

## Save and download results
This contains checkpoints, optimizer/RNG state, configs, logs, batch benchmark,
MIDI/actions, listening clips and partner probes. It excludes base model weights
and the original dataset. Keep the original package plus the results ZIP to resume.

In [ ]:
results_zip = RESULTS_ROOT.parent / "a100_music_results_v2.zip"
run_command(["h100_assets.py", "export", "--run", RESULTS_ROOT, "--destination", results_zip])
display(FileLink(str(results_zip)))
# Colab only, if FileLink is inconvenient:
# from google.colab import files
# files.download(str(results_zip))